In [2]:
#!/usr/bin/env/python3
import time
import rospy
from iai_apartment_kitchen_msgs.srv import Authenticateuser

from pycram.designators.specialized_designators.location.giskard_location import GiskardLocation
from pycram.worlds.bullet_world import BulletWorld
from pycram.designators.action_designator import *
from pycram.designators.location_designator import *
from pycram.designators.object_designator import *
from pycram.datastructures.enums import ObjectType, WorldMode,Arms
from pycram.datastructures.pose import Pose
from pycram.process_module import simulated_robot, with_simulated_robot, real_robot,with_real_robot
from pycram.object_descriptors.urdf import ObjectDescription
from pycram.world_concepts.world_object import Object
from pycram.datastructures.dataclasses import Color
from pycram.ros_utils.robot_state_updater import WorldStateUpdater
import pycrap
from pycram.pose_generator_and_validator import PoseGenerator
from pycram.ros_utils.tf_broadcaster import TFBroadcaster
from pycram.local_transformer import LocalTransformer
from pycram.external_interfaces import giskard
from pycrap.ontologies import *

[WARN] [1744791268.126250]: [helper.py:91:get_robot_description_path] Could not find Multiverse resources path and no other resources were given.
Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
[WARN] [1744791268.398323]: [ur5e_controlled_description.py:48:<module>] Could not initialize ur5e description as Multiverse resources path not found.
Unknown attribute "spring_reference" in /robot[@name='stretch_description']/joint[@name='joint_right_wheel']/dynamics
Unknown attribute "spring_stiffness" in /robot[@name='stretch_description']/joint[@name='joint_right_wheel']/dynamics
Unknown attribute "spring_reference" in /robot[@name='stretch_description']/joint[@name='joint_left_wheel']/dynamics
Unknown attribut

In [3]:
world = BulletWorld(mode=WorldMode.GUI)
stretch= Object('stretch_description',Robot,'stretch_description.urdf')
stretch_designator = ObjectDesignatorDescription(names=['stretch_description']).resolve()
apartment=Object('apartment',Apartment,'apartment.urdf')


# r = WorldStateUpdater("/tf", "/joint_states")

lt = LocalTransformer()

# table = Object("table", pycrap.Genobj, "big_table.stl", pose=Pose([14.55, 1.05, 0.35], [0, 0, 0, 1]))
# coffee_table = Object("cofee_table", ObjectType.GENERIC_OBJECT, "coffee_table.stl",
#                       pose=Pose([12.75, 3.55, 0], [0, 0, 1, 1]))
# armchair = Object("armchair", ObjectType.GENERIC_OBJECT, "armchair_lowres.stl",
#                   pose=Pose([14.1, 3.45, 0.355], [0, 0, -1, 1]))
# sofa = Object("sofa", ObjectType.GENERIC_OBJECT, "sofa_lowres.stl", pose=Pose([12.8, 4.75, 0.355], [0, 0, 0, 1]))


retrieve_arm_fully = MoveJointsMotion(["joint_arm_l1", "joint_arm_l2", "joint_arm_l3", "joint_arm_l0",
                                       "joint_lift"],
                                      [0., 0., 0., 0., 1.1])

#milk = Object("milk", pycrap.Milk, "milk.stl", pose=Pose([11.5, 2.0, 1.02]),
 #             color=Color(1, 0, 0, 1))
#jeroen_cup=Object("jeroen_cup",pycrap.Cup,"jeroen_cup.stl",pose=Pose([13.1,0.73,0.8],[0,0,1,0]),color=Color(0, 0, 1, 1))

#bowl=Object("bowl",pycrap.Bowl,"bowl.stl",pose=Pose([2.5,2.3,0.86]))
#apartment.attach(bowl,"cabinet10_drawer_top")


spoon=Object("spoon",Spoon,"spoon.stl",pose=Pose([2.55,1.6,0.85],[0,0,0,1]),color=Color(0, 1, 0, 1))
#apartment.attach(spoon,"cabinet11_drawer_top")

WorldStateUpdater(tf_topic="/tf", joint_state_topic="/joint_states")

[INFO] [1744791274.269238]: [cache_manager.py:105:look_for_file_in_data_dir] Found file plane.urdf in /home/stretch/ros_ws/pycram_ws/src/pycram/resources/objects/plane.urdf
[INFO] [1744791274.445558]: [cache_manager.py:105:look_for_file_in_data_dir] Found file stretch_description.urdf in /home/stretch/ros_ws/pycram_ws/src/pycram/resources/robots/stretch_description.urdf


Unknown attribute "spring_reference" in /robot[@name='stretch_description']/joint[@name='joint_right_wheel']/dynamics
Unknown attribute "spring_stiffness" in /robot[@name='stretch_description']/joint[@name='joint_right_wheel']/dynamics
Unknown attribute "spring_reference" in /robot[@name='stretch_description']/joint[@name='joint_left_wheel']/dynamics
Unknown attribute "spring_stiffness" in /robot[@name='stretch_description']/joint[@name='joint_left_wheel']/dynamics
Unknown tag "surface" in /robot[@name='stretch_description']/link[@name='caster_link']/collision[1]


[INFO] [1744791281.955320]: [cache_manager.py:105:look_for_file_in_data_dir] Found file apartment.urdf in /home/stretch/ros_ws/pycram_ws/src/pycram/resources/objects/apartment.urdf


Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


[INFO] [1744791299.207355]: [cache_manager.py:105:look_for_file_in_data_dir] Found file spoon.stl in /home/stretch/ros_ws/pycram_ws/src/pycram/resources/objects/spoon.stl


Unknown tag "material" in /robot[@name='spoon_object']/link[@name='spoon_main']/collision[1]


In [3]:
%load_ext autoreload
%autoreload 2

#cup_desig=BelieveObject(types=[pycrap.Cup])
#milk_desig=BelieveObject(types=[pycrap.Milk])
apartment_desig=BelieveObject(types=[Apartment])
robot_desig=BelieveObject(names=["stretch_description"])
spoon_desig=BelieveObject(types=[Spoon])




open_gripper=MoveGripperMotion(GripperState.OPEN,Arms.RIGHT)
close_gripper=MoveGripperMotion(GripperState.CLOSE,Arms.RIGHT)
target=Pose([9,3,1])
#there is no head_pan_link in stretch description, and it was used in process_modules.
#LookingMotion(milk.pose)
lokdown=Pose([10,1.9,0.4])
#look_at=LookAtAction([jeroen_cup.pose]).resolve()
nav=MoveMotion(Pose([10,2,0],[0,0,0,1]))
#print(stretch.pose)

open_arm=MoveJointsMotion(['joint_arm_l0', 'joint_arm_l1', 'joint_arm_l2', 'joint_arm_l3', 'joint_arm_l4'],
                     [0., 0.1, 0., 0., 0.])

retrieve_arm_fully2 = MoveJointsMotion(["joint_arm_l1", "joint_arm_l2", "joint_arm_l3", "joint_arm_l0",
                                       "joint_lift"],
                                      [0., 0., 0., 0.1, 0.8])

#detect_cup=DetectAction(cup_desig)
#detect_cup=DetectingMotion(technique=DetectionTechnique.TYPES,state=DetectionState.START, object_designator_description=cup_desig)
#print(jeroen_cup)
#print(cup_desig.resolve().world_object)
o =ObjectDesignatorDescription()
# query_object(cup_desig)
#found_objects = [obj for obj in world.objects if obj.tf_frame == "jeroen_cup"]
#World.robot.get_link_tf_frame("base_link")
#st=stretch_orientation_generator()
local_transformer = LocalTransformer()
l = LocalTransformer()
test_pose = Pose([2, 2, 1], [0, 0, 0, 1], "map")

approach_pose=Pose([9.9,3.08,0],[0,0,1,0])
#goal_pose=det.pose
tip_link="link_grasp_center"
root_link="map"
robot_base_link="base_link"

Exception in thread /joint_states:
Traceback (most recent call last):
  File "/usr/lib/python3.8/threading.py", line 932, in _bootstrap_inner
    self.run()
  File "/home/stretch/.local/lib/python3.8/site-packages/ipykernel/ipkernel.py", line 761, in run_closure
    _threading_Thread_run(self)
  File "/usr/lib/python3.8/threading.py", line 870, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/ros/noetic/lib/python3/dist-packages/rospy/impl/tcpros_pubsub.py", line 185, in robust_connect_subscriber
    conn.receive_loop(receive_cb)	    
  File "/opt/ros/noetic/lib/python3/dist-packages/rospy/impl/tcpros_base.py", line 846, in receive_loop
    self.close()
  File "/opt/ros/noetic/lib/python3/dist-packages/rospy/impl/tcpros_base.py", line 858, in close
    self.socket.close()
AttributeError: 'NoneType' object has no attribute 'close'


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
str_pose=stretch.pose
str_pose

header: 
  seq: 0
  stamp: 
    secs: 1741601001
    nsecs: 550773143
  frame_id: "map"
pose: 
  position: 
    x: 9.981669173751055
    y: 1.9627734933866594
    z: 0.0
  orientation: 
    x: 0.0
    y: 0.0
    z: 0.021272100214164986
    w: 0.9997737232756613

In [5]:
with real_robot:
    det=detect_cup.perform()
    print(det.pose)

    #copy of the detected pose
    det_pose_copy=det.pose.copy()
    det_pose_copy.pose.position.y -= 0.1
    det_pose_copy.pose.position.x -= 0.03
    det_pose_copy.pose.position.z += 0.1
    print("copy: ")
    print(det_pose_copy)

    #LookingMotion(det.pose).perform()

    print("---------------------------------------------------------------")

    #cost_loc=CostmapLocation(det,reachable_for=stretch_designator,reachable_arm=Arms.RIGHT).resolve()
   # print(cost_loc)




[INFO] [1741597485.529346]: [robokudo.py:73:wrapper] Successfully initialized Robokudo interface
[INFO] [1741597487.789706]: [robokudo.py:76:wrapper] Waiting for action server
[INFO] [1741597488.266260]: [robokudo.py:78:wrapper] Action server is available
blue
[INFO] [1741597488.488303]: [robokudo.py:116:active_callback] Goal is now being processed by the action server
[INFO] [1741597488.492420]: [robokudo.py:122:send_query] Goal has been sent to the action server
[INFO] [1741597488.692268]: [robokudo.py:113:done_callback] Query completed with state: 3
[INFO] [1741597488.935963]: [robokudo.py:125:send_query] Waiting for result from the action server
header: 
  seq: 0
  stamp: 
    secs: 1741597489
    nsecs:  80893516
  frame_id: "map"
pose: 
  position: 
    x: 13.114548749923282
    y: 0.6355258549908426
    z: 0.925955892650566
  orientation: 
    x: 0.0
    y: 0.0
    z: -0.7648684165682519
    w: 0.6441865454481143
copy: 
header: 
  seq: 0
  stamp: 
    secs: 1741597489
    nsecs:

Unknown attribute "spring_reference" in /robot[@name='stretch_description']/joint[@name='joint_right_wheel']/dynamics
Unknown attribute "spring_stiffness" in /robot[@name='stretch_description']/joint[@name='joint_right_wheel']/dynamics
Unknown attribute "spring_reference" in /robot[@name='stretch_description']/joint[@name='joint_left_wheel']/dynamics
Unknown attribute "spring_stiffness" in /robot[@name='stretch_description']/joint[@name='joint_left_wheel']/dynamics
Unknown tag "surface" in /robot[@name='stretch_description']/link[@name='caster_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]
Unknown tag "material" in /robot[@name='jeroen_cup_object']/link[@name='jeroen_cup_main']/collision[1]


poses of the gripper: header: 
  seq: 0
  stamp: 
    secs: 1741597543
    nsecs: 653351068
  frame_id: "map"
pose: 
  position: 
    x: 13.095553398132324
    y: 0.5442771315574646
    z: 1.045580506324768
  orientation: 
    x: -0.009865073904778712
    y: -0.005182322776168228
    z: -0.7554316127313131
    w: 0.6551327364234398
target_map is --> : 0.02414151166791238
CostmapLocation.Location(pose=header: 
  seq: 0
  stamp: 
    secs: 1741597543
    nsecs: 649080514
  frame_id: "map"
pose: 
  position: 
    x: 13.439074811219523
    y: 0.9753960805161985
    z: 0.0
  orientation: 
    x: 0.0
    y: 0.0
    z: -0.4816906300484136
    w: 0.8763413358523963, reachable_arms=[<Arms.RIGHT: 1>], tried_grasps=[<Grasp.FRONT: 0>])


In [28]:
stretch.set_pose(str_pose)

In [5]:
#demo for picking the cup from table

@with_real_robot
def detect(cup_desig):
    detect_cup=DetectingMotion(technique=DetectionTechnique.TYPES,state=DetectionState.START, object_designator_description=cup_desig)
    det=detect_cup.perform()
    return det
@with_real_robot
def change_pose():
    print("changing pose.....")
    NavigateAction([Pose([stretch.pose.position.x+2,1.6,0],[0,0,-1,1])]).resolve().perform()
    stretch.set_pose(Pose([stretch.pose.position.x+2,1.6,0],[0,0,-1,1]))


@with_real_robot
def pickup():
    max=3
    attempts=0
    det=None


    ParkArmsAction([Arms.RIGHT]).resolve().perform()

    NavigateAction([Pose([13.2,1.6,0],[0,0,-1,1])]).resolve().perform()
    stretch.set_pose(Pose([13.2,1.6,0],[0,0,-1,1]))
    time.sleep(2)


    while attempts < max:
        try:
            det=detect(cup_desig)

            if det:
                print(det.pose)
                NavigateAction([Pose([det.pose.position.x+0.03,det.pose.position.y+0.8,0],[0,0,0,1])]).resolve().perform()

                LookAtAction([Pose([-det.pose.position.x,det.pose.position.y,det.pose.position.z])]).resolve().perform()

                NavigateAction([Pose([det.pose.position.x+0.015,det.pose.position.y+0.7,0],[0,0,0,1])]).resolve().perform()
                MoveGripperMotion(GripperState.OPEN,Arms.RIGHT).perform()

                MoveJointsMotion(["joint_lift"], [det.pose.position.z-0.12]).perform()
                MoveJointsMotion(["joint_arm_l1", "joint_arm_l2", "joint_arm_l3"], [0.1, 0.1, 0.1]).perform()

                MoveGripperMotion(GripperState.CLOSE,Arms.RIGHT).perform()
                MoveJointsMotion(["joint_arm_l1", "joint_arm_l2", "joint_arm_l3"], [0, 0, 0]).perform()
                LookingMotion(target=Pose([13,0,1])).perform()

                #LookAtAction([Pose([16,1,1.2])]).resolve().perform()
                ParkArmsAction([Arms.RIGHT]).resolve().perform()


                #NavigateAction([Pose([10,2.2,0],[0,0,1,0])]).resolve().perform()



                NavigateAction([Pose([4,2.4,0],[0,0,1,0])]).resolve().perform()
                NavigateAction([Pose([3.65,2.6,0],[0,0,-1,1])]).resolve().perform()

                #LookAtAction([Pose([-2.9, 2.8, 1.1])]).resolve().perform()
                ParkArmsAction([Arms.RIGHT]).resolve().perform()
                MoveTCPMotion(target=Pose([2.86, 2.6, 1.05],
                                              [0, 0, 1, 0]), arm=Arms.RIGHT,
                                   allow_gripper_collision=True).perform()
                time.sleep(2)
                MoveGripperMotion(GripperState.OPEN,Arms.RIGHT).perform()

                ParkArmsAction([Arms.RIGHT]).resolve().perform()


                MoveGripperMotion(GripperState.CLOSE,Arms.RIGHT).perform()
                #LookAtAction([Pose([4.5,1.2,1.3])]).resolve().perform()

                ParkArmsAction([Arms.RIGHT]).resolve().perform()
                NavigateAction([Pose([10,2.65,0],[0,0,1,0])]).resolve().perform()
                break
            #else:
              #  print("no cup detected.")
                #attempts+=1
        except Exception as e:
            print(f"Error encountered: {e}. Retrying...")
            attempts+=1
            if attempts<max:
                    print("changing pose.....")
                    NavigateAction([Pose([stretch.pose.position.x+0.5,1.6,0],[0,0,-1,1])]).resolve().perform()
                    stretch.set_pose(Pose([stretch.pose.position.x+0.5,1.6,0],[0,0,-1,1]))
    if not det:
        print("Max attempts reached.No cup detected.")

In [22]:
with real_robot:
    #NavigateAction([Pose([13.6,1,0],[0,0,-1,1])]).resolve().perform()
    detect_cup=DetectingMotion(technique=DetectionTechnique.TYPES,state=DetectionState.START, object_designator_description=spoon_desig)
    det=detect_cup.perform()
    print(det.pose)
    gis_loc=GiskardLocation(det.pose,reachable_for=stretch_designator,reachable_arm=Arms.RIGHT).resolve()
    print(gis_loc)



In [20]:
with real_robot:
    #same y
    #NavigateAction([Pose([9.8,3.6,0],[0,0,-1,1])]).resolve().perform()
   # MoveJointsMotion(['joint_wrist_pitch'], [-1.7]).perform()
   # MoveJointsMotion(["joint_arm_l1", "joint_arm_l2"], [0.1, 0.1]).perform()
   # MoveGripperMotion(GripperState.OPEN,Arms.RIGHT).perform()
   # MoveJointsMotion(["joint_lift"], [0.94]).perform()

    MoveGripperMotion(GripperState.CLOSE,Arms.RIGHT).perform()
    #ParkArmsAction([Arms.RIGHT]).resolve().perform()



In [7]:
#demo for picking spoon inside drawer
def call_blum_service(command,argument):
    try:
        open_drawer=rospy.ServiceProxy("/blum_kitchen_server",Authenticateuser)
        response=open_drawer(command,argument)
        print(response)
    except rospy.ServiceException as err:
        rospy.logerr("Service call failed: %s"%err)
#rospy.wait_for_service("/blum_kitchen_server")
#call_blum_service("open","Kochbesteck")

@with_real_robot
def detect(spoon_design):
    detect_spoon=DetectingMotion(technique=DetectionTechnique.TYPES,state=DetectionState.START, object_designator_description=spoon_design)
    det=detect_spoon.perform()
    return det


@with_real_robot
def pick_from_drawer():
    det=None
    #need to place stretch behind the counter
    ParkArmsActionDescription(Arms.RIGHT).resolve().perform()

    NavigateActionDescription([Pose([1.74,1.6,0],[0,0,0,1])]).resolve().perform()
    stretch.set_pose(Pose([1.74,1.6,0],[0,0,0,1]))


    MoveJointsMotion(["joint_lift"], [1.1]).perform()

    #call blum api
    #apartment.set_joint_position('cabinet11_drawer_top_joint', 0.07)
    rospy.wait_for_service("/blum_kitchen_server")
    call_blum_service("open","Kochbesteck")
    time.sleep(2)



    #need to adjust the cam to look in the direction of the drawer
    LookAtActionDescription([Pose([2.1, 1.6, 0.85])]).resolve().perform()
    ParkArmsActionDescription(Arms.RIGHT).resolve().perform()

    try:
        #try to detect the spoon inside the drawer
        for i in range(2):
            det=detect(spoon_desig)
            print(f"try detect {i} , det is {det}")
    except Exception as e:
        print(f"Error encountered: {e}. Retrying...")


    if det is not None:
        print(det.pose)
        NavigateActionDescription([Pose([1.6,1.7,0],[0,0,0,1])]).resolve().perform()
        offset=1.547-round(det.pose.position.y,3)
        print(f"offset is {offset}")
        y=round(det.pose.position.y+offset,3)
        print(f"y is {y}")
        NavigateActionDescription([Pose([1.4,y,0],[0,0,1,1])]).resolve().perform()



        LookAtActionDescription([Pose([2, 0.5, 0.8])]).resolve().perform()
        ParkArmsActionDescription(Arms.RIGHT).resolve().perform()

        MoveJointsMotion(["joint_lift"], [1.1]).perform()
        time.sleep(1)
        MoveJointsMotion(['joint_wrist_pitch'], [-1.4]).perform()
        MoveJointsMotion(['joint_wrist_pitch'], [-1.52]).perform()
        time.sleep(2)


        MoveJointsMotion(["joint_arm_l1", "joint_arm_l2","joint_arm_l3"], [0.1,0.1,0.1]).perform()
        time.sleep(2)
        #MoveJointsMotion(['joint_gripper_finger_left', 'joint_gripper_finger_right'], [0.05, 0.05]).perform()
        MoveGripperMotion(GripperState.OPEN,Arms.RIGHT).perform()
        time.sleep(1)
       # MoveJointsMotion(["joint_arm_l1", "joint_arm_l2"], [0.05,0.05]).perform()
        #time.sleep(2)



        MoveJointsMotion(["joint_lift"], [0.959]).perform()

        MoveGripperMotion(GripperState.CLOSE,Arms.RIGHT).perform()
        MoveJointsMotion(["joint_lift"], [1.1]).perform()


        time.sleep(2)


        MoveJointsMotion(["joint_arm_l1", "joint_arm_l2","joint_arm_l3"], [0,0,0]).perform()

        #NavigateAction([Pose([1.5,1.55,0],[0,0,1,1])]).resolve().perform()

        time.sleep(2)
        MoveJointsMotion(['joint_wrist_yaw'],[1.5]).perform()
        time.sleep(2)
        MoveJointsMotion(["joint_arm_l1", "joint_arm_l2","joint_arm_l3"], [0,0,0]).perform()
        MoveJointsMotion(["joint_lift"], [0.9]).perform()
        time.sleep(2)
        MoveJointsMotion(["joint_arm_l0","joint_arm_l1", "joint_arm_l2","joint_arm_l3"], [0.1,0.1,0.1,0.1]).perform()
        time.sleep(2)
        MoveJointsMotion(["joint_arm_l0","joint_arm_l1", "joint_arm_l2","joint_arm_l3"], [0,0,0,0]).perform()
        #time.sleep(2)
        MoveJointsMotion(["joint_lift"], [1.1]).perform()
        LookAtActionDescription([Pose([4, 1.8, 1])]).resolve().perform()
        MoveJointsMotion(["joint_arm_l0","joint_arm_l1", "joint_arm_l2","joint_arm_l3"], [0,0,0,0]).perform()
    else:
        print("No SPOON detected.")






In [5]:
with real_robot:
    NavigateActionDescription([Pose([1.74, 1.2 ,0],[0,0,1,1])]).resolve().perform()
    #LookAtActionDescription([Pose([2, 2, 1])]).resolve().perform()

   # detect_spoon=DetectingMotion(technique=DetectionTechnique.TYPES,state=DetectionState.START, object_designator_description=spoon_desig)
    #det=detect_spoon.perform()
    #print(det)

    #pick_from_drawer()
    # MoveGripperMotion(GripperState.OPEN,Arms.RIGHT).perform()
    #MoveJointsMotion(["joint_lift"], [1.1]).perform()
    #MoveJointsMotion(["joint_arm_l1", "joint_arm_l2","joint_arm_l3"], [0,0,0]).perform()
    #MoveJointsMotion(['joint_wrist_pitch'], [0]).perform()

NavigationGoalNotReachedError: Navigation goal not reached. Current pose: Pose: [1.75, 1.162, 0.0], [0.0, 0.0, 0.745, 0.667] in frame map, goal pose: Pose: [1.74, 1.2, 0], [0.0, 0.0, 0.707, 0.707] in frame map

In [8]:
world.robot.get_pose()

Pose: [1.635, 0.643, 0.0], [0.0, 0.0, 0.719, 0.695] in frame map

In [62]:
with real_robot:
 #   MoveJointsMotion(["joint_arm_l1", "joint_arm_l2","joint_arm_l3"], [0.1,0.1,0.1]).perform()
  #  MoveJointsMotion(['joint_wrist_pitch'], [0]).perform()
   # ParkArmsAction([Arms.RIGHT]).resolve().perform()
   # MoveJointsMotion(["joint_lift"], [1.1]).perform()
   #NavigateAction([Pose([1.5,1.82,0],[0,0,1,1])]).resolve().perform()
    MoveJointsMotion(["joint_arm_l1", "joint_arm_l2","joint_arm_l3"], [0,0,0]).perform()
    time.sleep(2)
    MoveJointsMotion(['joint_wrist_yaw'],[1.5]).perform()
    time.sleep(2)
    MoveJointsMotion(["joint_lift"], [0.99]).perform()
    time.sleep(2)
    MoveJointsMotion(["joint_arm_l1", "joint_arm_l2","joint_arm_l3","joint_arm_l3"], [0.1,0.1,0.1,0.07]).perform()


In [3]:
with real_robot:
    # MoveGripperMotion(GripperState.CLOSE,Arms.RIGHT).perform()
    #MoveTorsoActionDescription([TorsoState.HIGH]).resolve().perform()
    ParkArmsActionDescription(Arms.RIGHT).resolve().perform()


[INFO] [1744787808.353376]: [giskard.py:84:wrapper] Successfully initialized Giskard interface


Unknown attribute "spring_reference" in /robot[@name='stretch_description']/joint[@name='joint_right_wheel']/dynamics
Unknown attribute "spring_stiffness" in /robot[@name='stretch_description']/joint[@name='joint_right_wheel']/dynamics
Unknown attribute "spring_reference" in /robot[@name='stretch_description']/joint[@name='joint_left_wheel']/dynamics
Unknown attribute "spring_stiffness" in /robot[@name='stretch_description']/joint[@name='joint_left_wheel']/dynamics
Unknown tag "surface" in /robot[@name='stretch_description']/link[@name='caster_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]
Unknown tag "material" in /robot[@name='spoon_object']/link[@name='spoon_main']/collision[1]


[INFO] [1744787813.070866]: [giskard.py:181:spawn_object] GiskardSpawnURDF Return value: error: 
  code: 0
  msg: '' ObjectName:apartment


In [6]:
stretch.set_pose(Pose([2,3,0],[0,0,0,1]))

In [30]:
World.robot.get_link_tf_frame("link_lift")

'stretch_description/link_lift'